# Radiation Damage from a Primary Knock-On Atom (PKA)

This notebook analyzes the small radiation-damage simulations run in Atomify.

It has **two jobs**:

1. Analyze the **time history of the current PKA simulation** from `pka_history.csv`.
2. Combine final results from several PKA energies to estimate an **apparent threshold displacement energy** for the teaching model.

The notebook deliberately uses only Python's standard library, NumPy, and Matplotlib so it will work well in Atomify's browser-based Jupyter environment.


## 1. Simulation settings

Edit these values so they match the LAMMPS simulation you just ran.

`DIRECTION` is a label only; for example `"110"` means the PKA was launched along [110].


In [ ]:
EPKA_eV = 150
DIRECTION = "110"
PKA_SPECIES = "O"
DISPLACEMENT_CUTOFF_A = 1.0
HISTORY_FILE = "pka_history.csv"


## 2. Load the current simulation history

The LAMMPS input should write a simple comma-separated file with columns:

`step,time_ps,msd_A2,ndisplaced,maxdisp_A`


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def read_pka_history(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"{filename!r} was not found. Run the PKA simulation first and make "
            "sure the LAMMPS input writes pka_history.csv."
        )
    rows = []
    with open(filename, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#") or line.lower().startswith("step,"):
                continue
            parts = [x.strip() for x in line.split(",")]
            if len(parts) != 5:
                continue
            try:
                rows.append((
                    int(float(parts[0])), float(parts[1]), float(parts[2]),
                    int(float(parts[3])), float(parts[4])
                ))
            except ValueError:
                continue
    if not rows:
        raise ValueError(f"No usable data rows were found in {filename!r}.")
    data = np.array(rows, dtype=float)
    return {
        "step": data[:,0].astype(int),
        "time_ps": data[:,1],
        "msd_A2": data[:,2],
        "ndisplaced": data[:,3].astype(int),
        "maxdisp_A": data[:,4],
    }

history = read_pka_history(HISTORY_FILE)
print(f"Loaded {len(history['step'])} frames from {HISTORY_FILE}")
print(f"Time span: {history['time_ps'][0]:.4f} to {history['time_ps'][-1]:.4f} ps")


## 3. Summarize peak and residual damage

**Peak damage** tells us how violent the collision event became.  
**Residual damage** tells us what remains at the end of the simulation.


In [ ]:
peak_nd = int(np.max(history["ndisplaced"]))
final_nd = int(history["ndisplaced"][-1])
peak_msd = float(np.max(history["msd_A2"]))
final_msd = float(history["msd_A2"][-1])
peak_maxdisp = float(np.max(history["maxdisp_A"]))
final_maxdisp = float(history["maxdisp_A"][-1])
peak_i = int(np.argmax(history["ndisplaced"]))
peak_time_ps = float(history["time_ps"][peak_i])

print("PKA RADIATION-DAMAGE SUMMARY")
print("----------------------------")
print(f"Species                  : {PKA_SPECIES}")
print(f"PKA energy               : {EPKA_eV:g} eV")
print(f"Direction                : [{DIRECTION}]")
print(f"Displacement cutoff      : {DISPLACEMENT_CUTOFF_A:g} A")
print(f"Peak displaced atoms     : {peak_nd}")
print(f"Peak-damage time         : {peak_time_ps:.4f} ps")
print(f"Residual displaced atoms : {final_nd}")
print(f"Peak MSD                 : {peak_msd:.5f} A^2")
print(f"Final MSD                : {final_msd:.5f} A^2")
print(f"Peak maximum displacement: {peak_maxdisp:.5f} A")
print(f"Final maximum displacement: {final_maxdisp:.5f} A")
print()
print("Interpretation:", "persistent displacement damage remains." if final_nd else "no atoms remain beyond the displacement cutoff.")


## 4. Damage evolution during the collision


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["time_ps"], history["msd_A2"])
plt.xlabel("Time (ps)")
plt.ylabel("Mean-square displacement (A^2)")
plt.title(f"MSD: {PKA_SPECIES} PKA, {EPKA_eV:g} eV, [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["time_ps"], history["ndisplaced"], marker="o", markersize=3)
plt.xlabel("Time (ps)")
plt.ylabel(f"Atoms displaced > {DISPLACEMENT_CUTOFF_A:g} A")
plt.title(f"Transient and residual damage: {EPKA_eV:g} eV [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["time_ps"], history["maxdisp_A"])
plt.xlabel("Time (ps)")
plt.ylabel("Maximum atomic displacement (A)")
plt.title(f"Largest displacement: {EPKA_eV:g} eV [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


## 5. Build an energy sweep

Atomify examples may reset run files when they are relaunched, so the safest class workflow is to copy the **final RESULT line** from each simulation into the list below.

Each entry is:

`(energy_eV, residual_displaced, final_MSD_A2, max_displacement_A)`


In [ ]:
runs = [
    # energy_eV, residual_displaced, final_MSD_A2, max_displacement_A
    (50,  0, 0.0220993057862484, 0.363060229310346),
    (150, 4, 0.0905781177608897, 4.11715777722114),
]

runs = sorted(runs, key=lambda r: r[0])

print("Energy (eV) | Residual displaced | Final MSD (A^2) | Max displacement (A)")
print("-"*76)
for e, nd, msd, md_ in runs:
    print(f"{e:11g} | {nd:18d} | {msd:15.5f} | {md_:20.5f}")


## 6. Estimate the apparent threshold displacement energy

For this introductory activity, define a run as "damaged" if at least one atom remains beyond the displacement cutoff at the end.

The notebook reports a **bracket**, not a falsely precise threshold.


In [ ]:
undamaged = [r[0] for r in runs if r[1] == 0]
damaged = [r[0] for r in runs if r[1] > 0]

if damaged:
    first_damage = min(damaged)
    lower_candidates = [e for e in undamaged if e < first_damage]
    if lower_candidates:
        last_no_damage = max(lower_candidates)
        print(f"Apparent threshold bracket for {PKA_SPECIES} PKA [{DIRECTION}]:")
        print(f"{last_no_damage:g} < E_d <= {first_damage:g} eV")
        print(f"Next useful simulations are between {last_no_damage:g} and {first_damage:g} eV.")
    else:
        print(f"Damage already occurs at {first_damage:g} eV. Test lower energies.")
else:
    print("No residual damage observed yet. Test higher PKA energies.")


## 7. Energy-sweep figures


In [ ]:
energies = np.array([r[0] for r in runs], dtype=float)
residual = np.array([r[1] for r in runs], dtype=float)

plt.figure(figsize=(7,4))
plt.plot(energies, residual, marker="o")
plt.xlabel("PKA energy (eV)")
plt.ylabel("Residual displaced atoms")
plt.title(f"Persistent damage vs PKA energy: {PKA_SPECIES} [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
maxdisp = np.array([r[3] for r in runs], dtype=float)

plt.figure(figsize=(7,4))
plt.plot(energies, maxdisp, marker="o")
plt.xlabel("PKA energy (eV)")
plt.ylabel("Maximum displacement (A)")
plt.title(f"Maximum displacement vs PKA energy: {PKA_SPECIES} [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
final_msd = np.array([r[2] for r in runs], dtype=float)

plt.figure(figsize=(7,4))
plt.plot(energies, final_msd, marker="o")
plt.xlabel("PKA energy (eV)")
plt.ylabel("Final MSD (A^2)")
plt.title(f"Final MSD vs PKA energy: {PKA_SPECIES} [{DIRECTION}]")
plt.grid(True, alpha=0.25)
plt.show()


## 8. Questions for the student report

1. At what energy does persistent displacement first appear?
2. Does maximum displacement change smoothly with PKA energy, or is there a sharp transition?
3. Compare peak displaced atoms with residual displaced atoms. What does the difference mean physically?
4. Why is global MSD less sensitive to a small number of strongly displaced atoms?
5. Repeat for another crystallographic direction. Does the apparent threshold change?
6. What limitations arise from the finite simulation cell and the interatomic potential?
7. Why should this exercise report an **apparent threshold** rather than a definitive experimental threshold displacement energy?
